# Day 3: Fine-Tuning, Evaluation & Deployment

**Workshop: Introduction to Generative AI & LLMs**

---

## Learning Objectives

By the end of this session, you will be able to:

1. Decide when to fine-tune vs. prompt engineering vs. RAG
2. Prepare data in JSONL format for fine-tuning
3. Understand LoRA/PEFT fine-tuning with HuggingFace Transformers
4. Evaluate LLM outputs using BLEU, ROUGE, and LLM-as-judge
5. Build and deploy a chat interface with Gradio
6. Apply ethical considerations for responsible AI use

---

## 1. When to Fine-Tune vs. Prompt vs. RAG

### Decision Framework

Choosing the right approach depends on your specific requirements:

```
                     START
                       |
              Do you need the model
              to learn NEW knowledge?
                 /           \
               Yes            No
               /                \
     Is the knowledge        Does the model need
     changing often?         a specific style/format?
       /        \              /           \
     Yes        No           Yes            No
      |          |            |              |
    [RAG]   [Fine-tune]  [Fine-tune     [Prompt
              or RAG      or Prompt]     Engineering]
```

### Comparison Table

| Criterion | Prompt Engineering | RAG | Fine-Tuning |
|-----------|-------------------|-----|-------------|
| **Setup cost** | None | Medium (vector DB) | High (GPU, data) |
| **New knowledge** | Limited to context | Excellent | Good (static) |
| **Style/format control** | Moderate | Moderate | Excellent |
| **Latency** | Low | Medium (retrieval) | Low |
| **Data requirement** | None | Documents | 100s-1000s examples |
| **Maintenance** | Easy | Medium | Hard (retrain) |
| **Hallucination risk** | Moderate | Low (grounded) | Moderate |
| **Cost per query** | Token-based | Token + retrieval | Lower (smaller model) |

### When to Fine-Tune

Fine-tuning is the right choice when:
- You need consistent output format/style that prompting cannot achieve
- You want to distill a large model's capabilities into a smaller, cheaper model
- You have domain-specific terminology or conventions (medical, legal, scientific)
- You need to reduce latency and cost by using a smaller specialized model
- You have labeled training data (hundreds to thousands of examples)

### When NOT to Fine-Tune

- Your knowledge base changes frequently (use RAG)
- You need factual accuracy on specific documents (use RAG)
- Prompt engineering achieves acceptable results (simplest approach)
- You have fewer than 50-100 examples

---

## 2. Setup

In [ ]:
# Install required packages
!pip install openai transformers peft accelerate datasets bitsandbytes --quiet
!pip install nltk rouge-score gradio --quiet

In [ ]:
import os
import json
import random
from textwrap import dedent
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print(f"OpenAI API key configured: {'Yes' if os.environ.get('OPENAI_API_KEY') else 'No'}")

---

## 3. Data Preparation for Fine-Tuning

Fine-tuning requires data in JSONL format (one JSON object per line). The format depends on the provider:

- **OpenAI**: `{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}`
- **HuggingFace**: Varies by trainer, but commonly `{"instruction": ..., "input": ..., "output": ...}`

In [ ]:
# Create training data for a "research paper summarizer" fine-tuning task

# Sample training data: paper abstract -> structured summary
training_examples = [
    {
        "messages": [
            {
                "role": "system",
                "content": "You are a research paper summarizer. Given a paper abstract, "
                           "produce a concise structured summary with: Main Contribution, "
                           "Method, Key Results, and Limitations."
            },
            {
                "role": "user",
                "content": "Summarize this abstract: We present a novel method for text "
                           "classification using graph neural networks. Our approach constructs "
                           "a word co-occurrence graph from the corpus and applies message "
                           "passing to learn node representations. Experiments on 5 benchmark "
                           "datasets show improvements of 1.5-3% over BERT baselines while "
                           "using 60% fewer parameters."
            },
            {
                "role": "assistant",
                "content": "**Main Contribution**: Novel GNN-based text classification method.\n"
                           "**Method**: Word co-occurrence graph + message passing for node representations.\n"
                           "**Key Results**: 1.5-3% improvement over BERT on 5 benchmarks with 60% fewer parameters.\n"
                           "**Limitations**: Relies on word co-occurrence which may miss semantic nuances; "
                           "graph construction adds preprocessing overhead."
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "You are a research paper summarizer. Given a paper abstract, "
                           "produce a concise structured summary with: Main Contribution, "
                           "Method, Key Results, and Limitations."
            },
            {
                "role": "user",
                "content": "Summarize this abstract: We introduce a continual learning framework "
                           "for large language models that mitigates catastrophic forgetting. "
                           "Our approach uses elastic weight consolidation combined with experience "
                           "replay to retain previously learned knowledge while adapting to new "
                           "tasks. We evaluate on a sequence of 10 NLP tasks and show that our "
                           "method retains 95% of performance on old tasks while achieving "
                           "comparable performance on new ones."
            },
            {
                "role": "assistant",
                "content": "**Main Contribution**: Continual learning framework for LLMs addressing catastrophic forgetting.\n"
                           "**Method**: Elastic weight consolidation + experience replay.\n"
                           "**Key Results**: Retains 95% performance on old tasks across a 10-task sequence.\n"
                           "**Limitations**: Experience replay requires storing old task data; "
                           "computational overhead scales with number of tasks."
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "You are a research paper summarizer. Given a paper abstract, "
                           "produce a concise structured summary with: Main Contribution, "
                           "Method, Key Results, and Limitations."
            },
            {
                "role": "user",
                "content": "Summarize this abstract: This paper proposes a multimodal fusion "
                           "architecture that combines text and image representations for "
                           "visual question answering. We use cross-attention between CLIP "
                           "image features and language model hidden states, enabling the model "
                           "to ground its text generation in visual content. Our approach "
                           "achieves state-of-the-art on VQAv2 and GQA benchmarks."
            },
            {
                "role": "assistant",
                "content": "**Main Contribution**: Multimodal fusion architecture for visual question answering.\n"
                           "**Method**: Cross-attention between CLIP image features and LM hidden states.\n"
                           "**Key Results**: State-of-the-art on VQAv2 and GQA benchmarks.\n"
                           "**Limitations**: Relies on CLIP's image representations which may miss fine-grained "
                           "visual details; cross-attention increases inference cost."
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "You are a research paper summarizer. Given a paper abstract, "
                           "produce a concise structured summary with: Main Contribution, "
                           "Method, Key Results, and Limitations."
            },
            {
                "role": "user",
                "content": "Summarize this abstract: We study the scaling laws governing "
                           "language model performance. Through systematic experiments with "
                           "models ranging from 10M to 10B parameters, we find that loss "
                           "follows a power law with respect to model size, dataset size, "
                           "and compute budget. These laws enable accurate prediction of "
                           "model performance before training, potentially saving millions "
                           "in compute costs."
            },
            {
                "role": "assistant",
                "content": "**Main Contribution**: Empirical scaling laws for language model performance prediction.\n"
                           "**Method**: Systematic experiments across model sizes (10M-10B parameters).\n"
                           "**Key Results**: Loss follows power law with model size, data size, and compute; "
                           "enables performance prediction before training.\n"
                           "**Limitations**: Laws may not generalize across architectures or modalities; "
                           "does not capture emergent capabilities at scale."
            }
        ]
    },
]

# Write to JSONL file
output_path = Path("training_data.jsonl")
with open(output_path, "w") as f:
    for example in training_examples:
        f.write(json.dumps(example) + "\n")

print(f"Training data written to {output_path}")
print(f"Number of examples: {len(training_examples)}")
print(f"\nSample entry (first 200 chars):")
print(json.dumps(training_examples[0])[:200] + "...")

In [ ]:
# Validate the JSONL file format

def validate_jsonl(filepath: str) -> dict:
    """Validate a JSONL file for fine-tuning."""
    errors = []
    stats = {"total": 0, "valid": 0, "roles": set()}

    with open(filepath) as f:
        for i, line in enumerate(f, 1):
            stats["total"] += 1
            try:
                data = json.loads(line)

                # Check required structure
                if "messages" not in data:
                    errors.append(f"Line {i}: Missing 'messages' key")
                    continue

                messages = data["messages"]
                if not isinstance(messages, list) or len(messages) < 2:
                    errors.append(f"Line {i}: 'messages' must be a list with at least 2 entries")
                    continue

                # Check roles
                roles = [m["role"] for m in messages]
                stats["roles"].update(roles)

                if "assistant" not in roles:
                    errors.append(f"Line {i}: Must have at least one 'assistant' message")
                    continue

                # Check each message has content
                for j, msg in enumerate(messages):
                    if "content" not in msg or not msg["content"].strip():
                        errors.append(f"Line {i}, message {j}: Empty content")

                stats["valid"] += 1

            except json.JSONDecodeError as e:
                errors.append(f"Line {i}: Invalid JSON - {e}")

    return {
        "stats": {**stats, "roles": list(stats["roles"])},
        "errors": errors,
        "is_valid": len(errors) == 0,
    }


result = validate_jsonl("training_data.jsonl")
print("Validation Results:")
print(f"  Total examples: {result['stats']['total']}")
print(f"  Valid examples: {result['stats']['valid']}")
print(f"  Roles found:    {result['stats']['roles']}")
print(f"  Is valid:       {result['is_valid']}")
if result["errors"]:
    print(f"  Errors:")
    for err in result["errors"]:
        print(f"    - {err}")

In [ ]:
# HuggingFace-style data preparation (Alpaca format)

hf_training_data = [
    {
        "instruction": "Summarize the following research paper abstract into a structured format "
                       "with Main Contribution, Method, Key Results, and Limitations.",
        "input": "We present a novel method for text classification using graph neural networks. "
                 "Our approach constructs a word co-occurrence graph from the corpus and applies "
                 "message passing to learn node representations.",
        "output": "**Main Contribution**: Novel GNN-based text classification method.\n"
                  "**Method**: Word co-occurrence graph + message passing.\n"
                  "**Key Results**: Improvements over BERT baselines.\n"
                  "**Limitations**: Graph construction overhead; may miss semantic nuances.",
    },
    # Additional examples would follow the same pattern
]

# Write HuggingFace format
with open("training_data_hf.jsonl", "w") as f:
    for example in hf_training_data:
        f.write(json.dumps(example) + "\n")

print("HuggingFace-format JSONL written.")
print("\nSample entry:")
print(json.dumps(hf_training_data[0], indent=2))

---

## 4. Fine-Tuning with HuggingFace Transformers (LoRA/PEFT)

Below is the complete code structure for fine-tuning a model using LoRA (Low-Rank Adaptation) with the PEFT library. This approach dramatically reduces memory requirements and training time.

**Note**: Running this code requires a GPU with at least 16GB VRAM. The cells below show the complete workflow but may not run on CPU-only machines.

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import load_dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# Fine-Tuning Configuration
# ============================================================

# Model to fine-tune (using a small model for demonstration)
MODEL_NAME = "microsoft/phi-2"  # 2.7B parameter model
# Alternatives: "mistralai/Mistral-7B-v0.1", "meta-llama/Llama-2-7b-hf"

# Quantization config for memory efficiency (4-bit quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute in float16
    bnb_4bit_use_double_quant=True,        # Double quantization for extra savings
)

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # Rank of the low-rank matrices
    lora_alpha=32,           # Scaling factor
    lora_dropout=0.05,       # Dropout for regularization
    target_modules=[         # Which layers to apply LoRA to
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

print("Configuration defined.")
print(f"\nLoRA Config:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {lora_config.target_modules}")

In [ ]:
# ============================================================
# Load Model and Tokenizer
# NOTE: This cell requires a GPU. It will download the model (~5GB)
# ============================================================

# Uncomment to run on GPU:

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"
#
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )
#
# # Prepare model for LoRA training
# model = prepare_model_for_kbit_training(model)
# model = get_peft_model(model, lora_config)
#
# # Print trainable parameters
# model.print_trainable_parameters()
# # Expected output: trainable params: ~4M (~0.15% of 2.7B)

print("[Model loading code shown above -- uncomment to run on GPU]")
print("\nWith LoRA (r=16), typical parameter counts:")
print("  Phi-2 (2.7B):    ~4M trainable (0.15%)")
print("  Mistral-7B:      ~10M trainable (0.14%)")
print("  LLaMA-2-13B:     ~20M trainable (0.15%)")

In [ ]:
# ============================================================
# Data Preprocessing for HuggingFace Trainer
# ============================================================

def format_instruction(example):
    """Format a single example into the model's expected input format."""
    if example.get("input"):
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return text


# Example of how to tokenize
sample = hf_training_data[0]
formatted = format_instruction(sample)
print("Formatted training example:")
print(formatted)

# In practice, you would tokenize like this:
# def tokenize(example):
#     text = format_instruction(example)
#     tokenized = tokenizer(
#         text,
#         truncation=True,
#         max_length=512,
#         padding="max_length",
#     )
#     tokenized["labels"] = tokenized["input_ids"].copy()
#     return tokenized

In [ ]:
# ============================================================
# Training Configuration and Execution
# ============================================================

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # Effective batch size = 4 * 4 = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,                          # Mixed precision training
    optim="paged_adamw_8bit",           # Memory-efficient optimizer
    report_to="none",                   # Disable wandb/tensorboard for workshop
)

print("Training arguments configured:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

# To actually train (uncomment on GPU):
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_dataset,
#     data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
# )
# trainer.train()
#
# # Save the LoRA adapter
# model.save_pretrained("./lora_adapter")
# tokenizer.save_pretrained("./lora_adapter")

print("\n[Training code shown above -- uncomment to run on GPU]")

In [ ]:
# ============================================================
# Loading and Using a Fine-Tuned LoRA Model
# ============================================================

# After training, load the adapter:
#
# from peft import PeftModel
#
# # Load base model
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
#
# # Load LoRA adapter on top
# model = PeftModel.from_pretrained(base_model, "./lora_adapter")
#
# # Generate with the fine-tuned model
# prompt = "### Instruction:\nSummarize this abstract...\n\n### Response:\n"
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.3)
# result = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(result)
#
# # Optional: Merge LoRA weights into base model for faster inference
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("./merged_model")

print("LoRA loading and inference code shown above.")
print("\nKey points:")
print("  1. LoRA adapters are tiny (~10-50MB) vs full models (GB)")
print("  2. You can swap adapters for different tasks on the same base model")
print("  3. merge_and_unload() creates a standalone model for deployment")

---

## 5. Evaluation Metrics

Evaluating LLM outputs requires both automated metrics and human/LLM judgment.

In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

print("NLTK and rouge-score loaded.")

In [ ]:
# ============================================================
# BLEU Score (for translation and short generation tasks)
# ============================================================

# BLEU measures n-gram overlap between generated and reference text
# Score range: 0.0 (no overlap) to 1.0 (perfect match)

reference = "The cat sat on the mat and looked at the window."
candidates = [
    "The cat sat on the mat and looked at the window.",     # Perfect match
    "The cat was sitting on the mat looking at the window.", # Paraphrase
    "A feline rested on the rug gazing at the glass.",      # Synonym-heavy
    "The dog played in the garden.",                         # Completely different
]

smoothie = SmoothingFunction().method1
ref_tokens = reference.lower().split()

print(f"Reference: \"{reference}\"")
print("=" * 60)

for candidate in candidates:
    cand_tokens = candidate.lower().split()

    # Compute BLEU with different n-gram weights
    bleu_1 = sentence_bleu([ref_tokens], cand_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu_2 = sentence_bleu([ref_tokens], cand_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    bleu_4 = sentence_bleu([ref_tokens], cand_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    print(f"\nCandidate: \"{candidate}\"")
    print(f"  BLEU-1: {bleu_1:.4f}  |  BLEU-2: {bleu_2:.4f}  |  BLEU-4: {bleu_4:.4f}")

In [ ]:
# ============================================================
# ROUGE Score (for summarization tasks)
# ============================================================

# ROUGE measures recall of n-grams from reference in the generated text
# - ROUGE-1: unigram overlap
# - ROUGE-2: bigram overlap
# - ROUGE-L: longest common subsequence

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

reference_summary = (
    "Large language models have shown remarkable capabilities in natural language "
    "understanding. Challenges include hallucination and bias. Researchers are "
    "working on alignment techniques to make models safer."
)

generated_summaries = [
    # Good summary
    "LLMs demonstrate strong NLU capabilities but face challenges like hallucination "
    "and bias. Alignment research aims to improve safety.",
    # Partially relevant
    "Language models are trained on large text corpora. They can perform many tasks "
    "including translation and summarization.",
    # Off-topic
    "Computer vision has made significant progress with convolutional neural networks "
    "and vision transformers.",
]

labels = ["Good summary", "Partial match", "Off-topic"]

print(f"Reference: \"{reference_summary[:80]}...\"")
print("=" * 70)

for label, summary in zip(labels, generated_summaries):
    scores = scorer.score(reference_summary, summary)

    print(f"\n[{label}] \"{summary[:60]}...\"")
    for metric, score in scores.items():
        print(f"  {metric}: Precision={score.precision:.3f} | Recall={score.recall:.3f} | F1={score.fmeasure:.3f}")

In [ ]:
# ============================================================
# Batch Evaluation Helper
# ============================================================

def evaluate_summaries(references: list[str], predictions: list[str]) -> dict:
    """Compute average ROUGE and BLEU scores across a set of examples."""
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    smoothie = SmoothingFunction().method1

    all_rouge1, all_rouge2, all_rougeL, all_bleu = [], [], [], []

    for ref, pred in zip(references, predictions):
        # ROUGE
        scores = scorer.score(ref, pred)
        all_rouge1.append(scores["rouge1"].fmeasure)
        all_rouge2.append(scores["rouge2"].fmeasure)
        all_rougeL.append(scores["rougeL"].fmeasure)

        # BLEU
        ref_tokens = ref.lower().split()
        pred_tokens = pred.lower().split()
        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie)
        all_bleu.append(bleu)

    return {
        "rouge1_f1": sum(all_rouge1) / len(all_rouge1),
        "rouge2_f1": sum(all_rouge2) / len(all_rouge2),
        "rougeL_f1": sum(all_rougeL) / len(all_rougeL),
        "bleu": sum(all_bleu) / len(all_bleu),
        "n_examples": len(references),
    }


# Test with our examples
refs = [reference_summary] * 3
preds = generated_summaries

results = evaluate_summaries(refs, preds)
print("Batch Evaluation Results:")
for k, v in results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

### 5.4 LLM-as-Judge Evaluation

Automated metrics like BLEU and ROUGE do not capture semantic quality well. Using an LLM to evaluate another LLM's output is increasingly popular and correlates better with human judgment.

In [ ]:
# LLM-as-Judge: Use GPT-4o-mini to evaluate quality

JUDGE_SYSTEM_PROMPT = dedent("""\
    You are an expert evaluator of AI-generated text. You will be given:
    - A task description
    - The original input
    - A generated output

    Evaluate the output on these criteria (score each 1-5):
    1. **Accuracy**: Is the information factually correct?
    2. **Completeness**: Does it cover the key points?
    3. **Clarity**: Is it well-written and easy to understand?
    4. **Conciseness**: Is it appropriately brief without being vague?

    Respond with a JSON object containing:
    - "accuracy": score (1-5)
    - "completeness": score (1-5)
    - "clarity": score (1-5)
    - "conciseness": score (1-5)
    - "overall": average score
    - "feedback": one sentence of constructive feedback""")


def llm_judge(task: str, input_text: str, output_text: str) -> dict:
    """Use an LLM to evaluate the quality of generated text."""
    user_msg = (
        f"Task: {task}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Generated Output:\n{output_text}"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )

    return json.loads(response.choices[0].message.content)


# Evaluate our summaries
task = "Summarize a research paper abstract into key points."
input_text = (
    "Large Language Models have demonstrated remarkable capabilities in natural language "
    "understanding and generation. Challenges remain around hallucination, bias, and "
    "environmental cost. Researchers are working on alignment techniques such as RLHF."
)

for label, summary in zip(labels, generated_summaries):
    print(f"\n{'='*60}")
    print(f"Evaluating [{label}]: \"{summary[:60]}...\"")
    judgment = llm_judge(task, input_text, summary)
    print(f"  Accuracy:     {judgment.get('accuracy', 'N/A')}/5")
    print(f"  Completeness: {judgment.get('completeness', 'N/A')}/5")
    print(f"  Clarity:      {judgment.get('clarity', 'N/A')}/5")
    print(f"  Conciseness:  {judgment.get('conciseness', 'N/A')}/5")
    print(f"  Overall:      {judgment.get('overall', 'N/A')}/5")
    print(f"  Feedback:     {judgment.get('feedback', 'N/A')}")

In [ ]:
# Pairwise comparison: Which output is better?

COMPARE_SYSTEM = dedent("""\
    You are an expert evaluator. Compare two AI-generated outputs for the same task.
    Determine which one is better and explain why.

    Respond with a JSON object:
    - "winner": "A" or "B" or "tie"
    - "reasoning": 2-3 sentences explaining your decision
    - "score_A": overall quality (1-10)
    - "score_B": overall quality (1-10)""")


def compare_outputs(task: str, input_text: str, output_a: str, output_b: str) -> dict:
    """Compare two outputs and determine which is better."""
    user_msg = (
        f"Task: {task}\n\n"
        f"Input:\n{input_text}\n\n"
        f"Output A:\n{output_a}\n\n"
        f"Output B:\n{output_b}"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": COMPARE_SYSTEM},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )

    return json.loads(response.choices[0].message.content)


# Compare the good summary vs partial match
comparison = compare_outputs(
    task="Summarize a research paper abstract",
    input_text=input_text,
    output_a=generated_summaries[0],
    output_b=generated_summaries[1],
)

print("Pairwise Comparison:")
print(f"  Winner: Output {comparison['winner']}")
print(f"  Score A: {comparison['score_A']}/10")
print(f"  Score B: {comparison['score_B']}/10")
print(f"  Reasoning: {comparison['reasoning']}")

---

## 6. Deployment with Gradio

Gradio makes it easy to create web-based interfaces for ML models. We will build a chat interface for a research assistant.

In [ ]:
import gradio as gr

print(f"Gradio version: {gr.__version__}")

In [ ]:
# ============================================================
# Simple Gradio Interface: Text Summarizer
# ============================================================

def summarize_text(text: str, style: str, max_length: str) -> str:
    """Summarize text using OpenAI API."""
    if not text.strip():
        return "Please enter some text to summarize."

    prompt = f"""Summarize the following text in {style} style.
Target length: {max_length}.

Text:
{text}

Summary:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert summarizer."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        max_tokens=500,
    )

    return response.choices[0].message.content


# Create the interface
summarizer_demo = gr.Interface(
    fn=summarize_text,
    inputs=[
        gr.Textbox(label="Input Text", lines=8, placeholder="Paste your text here..."),
        gr.Dropdown(
            choices=["concise", "academic", "bullet-point", "ELI5 (explain like I'm 5)"],
            value="concise",
            label="Summary Style",
        ),
        gr.Dropdown(
            choices=["1 sentence", "2-3 sentences", "1 paragraph", "3-5 bullet points"],
            value="2-3 sentences",
            label="Target Length",
        ),
    ],
    outputs=gr.Textbox(label="Summary", lines=5),
    title="AI Text Summarizer",
    description="Summarize any text using GPT-4o-mini. Choose your preferred style and length.",
    examples=[
        [
            "Large Language Models have demonstrated remarkable capabilities in natural language "
            "understanding and generation. These models can perform translation, summarization, "
            "and creative writing. Challenges remain around hallucination and bias.",
            "concise",
            "1 sentence",
        ],
    ],
)

# Launch (uncomment to run)
# summarizer_demo.launch(share=False)
print("Summarizer demo defined. Uncomment .launch() to start the web interface.")

In [ ]:
# ============================================================
# Gradio Chat Interface: Research Assistant
# ============================================================

RESEARCH_ASSISTANT_SYSTEM = dedent("""\
    You are an AI research assistant specializing in machine learning and NLP.

    Your capabilities:
    - Explain ML/AI concepts clearly at any level of expertise
    - Help brainstorm research ideas and methodologies
    - Assist with paper writing and literature review
    - Suggest relevant papers and resources
    - Debug code and explain algorithms

    Guidelines:
    - Be concise but thorough
    - Use examples when explaining concepts
    - Acknowledge uncertainty when appropriate
    - Suggest follow-up resources when relevant""")


def research_chat(message: str, history: list) -> str:
    """Chat function for the research assistant."""
    # Build messages from history
    messages = [{"role": "system", "content": RESEARCH_ASSISTANT_SYSTEM}]

    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7,
        max_tokens=1024,
    )

    return response.choices[0].message.content


# Create the chat interface
chat_demo = gr.ChatInterface(
    fn=research_chat,
    title="AI Research Assistant",
    description="Ask me anything about machine learning, NLP, and AI research!",
    examples=[
        "Explain the difference between BERT and GPT architectures.",
        "What are the best practices for fine-tuning LLMs with limited data?",
        "Help me design an experiment to compare RAG vs fine-tuning for Q&A.",
        "What evaluation metrics should I use for a text summarization model?",
    ],
    retry_btn="Retry",
    undo_btn="Undo",
    clear_btn="Clear",
)

# Launch (uncomment to run)
# chat_demo.launch(share=False)
print("Research Assistant chat demo defined. Uncomment .launch() to start.")

### Lab: Full Research Assistant with Gradio

Let's build a more complete application that combines multiple tools: summarization, Q&A, and paper analysis.

In [ ]:
# ============================================================
# Lab: Multi-Tab Research Assistant with Gradio
# ============================================================

def analyze_paper(abstract: str) -> str:
    """Analyze a paper abstract and provide structured feedback."""
    if not abstract.strip():
        return "Please paste a paper abstract."

    prompt = dedent(f"""\
        Analyze this research paper abstract and provide:

        1. **Main Contribution** (1-2 sentences)
        2. **Methodology** (brief description)
        3. **Strengths** (2-3 bullet points)
        4. **Potential Weaknesses** (2-3 bullet points)
        5. **Suggested Improvements** (2-3 bullet points)
        6. **Related Work** (suggest 2-3 relevant research directions)

        Abstract:
        {abstract}""")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an experienced peer reviewer."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.4,
        max_tokens=800,
    )

    return response.choices[0].message.content


def generate_research_ideas(topic: str, num_ideas: int) -> str:
    """Generate research ideas for a given topic."""
    if not topic.strip():
        return "Please enter a research topic."

    prompt = dedent(f"""\
        Generate {num_ideas} novel research ideas related to: {topic}

        For each idea, provide:
        - **Title**: A concise, descriptive title
        - **Motivation**: Why is this problem important? (2 sentences)
        - **Approach**: High-level methodology (2-3 sentences)
        - **Expected Impact**: What would success look like? (1-2 sentences)
        - **Feasibility**: Easy / Medium / Hard

        Focus on ideas that are novel, feasible, and impactful.""")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a creative AI research advisor."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.8,
        max_tokens=1500,
    )

    return response.choices[0].message.content


def explain_concept(concept: str, level: str) -> str:
    """Explain an ML/AI concept at the specified level."""
    if not concept.strip():
        return "Please enter a concept to explain."

    level_instructions = {
        "Beginner": "Explain as if to someone with no technical background. Use analogies and simple language.",
        "Intermediate": "Explain for someone with basic ML knowledge. Include some technical details.",
        "Advanced": "Explain in depth with mathematical formulations and implementation details.",
        "Expert": "Discuss at a research level, including recent advances, open problems, and nuances.",
    }

    prompt = dedent(f"""\
        Explain the concept of \"{concept}\" in machine learning / AI.

        Level: {level}
        {level_instructions[level]}

        Include:
        - Definition
        - How it works
        - Why it matters
        - A practical example""")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert ML educator."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.5,
        max_tokens=1000,
    )

    return response.choices[0].message.content


# Build the multi-tab Gradio app
with gr.Blocks(title="AI Research Assistant", theme=gr.themes.Soft()) as full_demo:
    gr.Markdown("# AI Research Assistant")
    gr.Markdown("A multi-tool assistant for ML/AI researchers.")

    with gr.Tab("Chat"):
        gr.Markdown("### Research Chat")
        chatbot = gr.Chatbot(height=400)
        msg = gr.Textbox(label="Your message", placeholder="Ask me anything about ML/AI...")
        clear = gr.ClearButton([msg, chatbot])

        def respond(message, chat_history):
            bot_message = research_chat(message, chat_history)
            chat_history.append((message, bot_message))
            return "", chat_history

        msg.submit(respond, [msg, chatbot], [msg, chatbot])

    with gr.Tab("Paper Analyzer"):
        gr.Markdown("### Analyze a Paper Abstract")
        abstract_input = gr.Textbox(
            label="Paper Abstract",
            lines=8,
            placeholder="Paste the abstract here...",
        )
        analyze_btn = gr.Button("Analyze", variant="primary")
        analysis_output = gr.Markdown(label="Analysis")
        analyze_btn.click(analyze_paper, inputs=abstract_input, outputs=analysis_output)

    with gr.Tab("Idea Generator"):
        gr.Markdown("### Generate Research Ideas")
        topic_input = gr.Textbox(label="Research Topic", placeholder="e.g., efficient RAG systems")
        num_ideas = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="Number of Ideas")
        generate_btn = gr.Button("Generate Ideas", variant="primary")
        ideas_output = gr.Markdown(label="Ideas")
        generate_btn.click(generate_research_ideas, inputs=[topic_input, num_ideas], outputs=ideas_output)

    with gr.Tab("Concept Explainer"):
        gr.Markdown("### Explain an ML/AI Concept")
        concept_input = gr.Textbox(label="Concept", placeholder="e.g., attention mechanism")
        level_input = gr.Radio(
            choices=["Beginner", "Intermediate", "Advanced", "Expert"],
            value="Intermediate",
            label="Explanation Level",
        )
        explain_btn = gr.Button("Explain", variant="primary")
        explanation_output = gr.Markdown(label="Explanation")
        explain_btn.click(explain_concept, inputs=[concept_input, level_input], outputs=explanation_output)

    with gr.Tab("Summarizer"):
        gr.Markdown("### Text Summarizer")
        sum_input = gr.Textbox(label="Text to Summarize", lines=8)
        sum_style = gr.Dropdown(
            choices=["concise", "academic", "bullet-point", "ELI5"],
            value="concise",
            label="Style",
        )
        sum_length = gr.Dropdown(
            choices=["1 sentence", "2-3 sentences", "1 paragraph", "3-5 bullet points"],
            value="2-3 sentences",
            label="Length",
        )
        sum_btn = gr.Button("Summarize", variant="primary")
        sum_output = gr.Textbox(label="Summary", lines=5)
        sum_btn.click(summarize_text, inputs=[sum_input, sum_style, sum_length], outputs=sum_output)


# Launch the full app (uncomment to run)
# full_demo.launch(share=False)
print("Full Research Assistant app defined with 5 tabs:")
print("  1. Chat - Conversational research assistant")
print("  2. Paper Analyzer - Structured abstract analysis")
print("  3. Idea Generator - Research idea brainstorming")
print("  4. Concept Explainer - Multi-level explanations")
print("  5. Summarizer - Text summarization")
print("\nUncomment full_demo.launch() to start the web interface.")

In [ ]:
# ============================================================
# Launch the app (uncomment one of these)
# ============================================================

# Option 1: Launch locally
# full_demo.launch(server_name="0.0.0.0", server_port=7860)

# Option 2: Launch with a public shareable link (72h)
# full_demo.launch(share=True)

# Option 3: Launch in Jupyter inline
# full_demo.launch(inline=True)

print("Uncomment one of the launch options above to start the app.")

---

## 7. Ethics & Responsible AI Use

### Key Ethical Considerations

#### 1. Hallucination and Factual Accuracy

LLMs can generate plausible-sounding but incorrect information. This is especially dangerous in:
- Medical advice
- Legal guidance
- Scientific claims
- Financial decisions

**Mitigation**: Use RAG for factual grounding, implement fact-checking pipelines, always disclose AI involvement.

#### 2. Bias and Fairness

LLMs inherit biases from their training data:
- Gender and racial stereotypes in text generation
- Cultural biases favoring Western perspectives
- Socioeconomic biases in language understanding

**Mitigation**: Evaluate outputs across demographic groups, use debiasing techniques, diversify training data.

#### 3. Privacy and Data Security

- Do not send sensitive or personally identifiable information (PII) to external APIs
- Be aware that API providers may log inputs for training
- Consider self-hosted models for sensitive applications
- Implement data anonymization before LLM processing

#### 4. Environmental Impact

- Training a single large LLM can emit as much CO2 as five cars over their lifetimes
- Inference at scale also has significant energy costs
- Use the smallest model that meets your needs
- Consider the environmental cost when choosing between approaches

#### 5. Academic Integrity

- Always disclose when AI tools are used in research
- LLMs should assist, not replace, human reasoning
- Verify all AI-generated citations (they may be fabricated)
- Follow your institution's AI use policies

#### 6. Transparency and Accountability

- Clearly label AI-generated content
- Maintain human oversight for critical decisions
- Document the AI systems and prompts used
- Establish clear lines of responsibility

### A Responsible AI Checklist

Before deploying an LLM-based system, consider:

- [ ] Have you tested for bias across different user groups?
- [ ] Is there a human review process for high-stakes outputs?
- [ ] Are users informed that they are interacting with AI?
- [ ] Is PII properly handled and protected?
- [ ] Are outputs logged for monitoring and improvement?
- [ ] Is there a feedback mechanism for users to report issues?
- [ ] Have you documented the system's limitations?
- [ ] Is the environmental impact proportional to the benefit?

---

## Summary

In this session, we covered:

1. **Decision Framework**: When to use prompt engineering vs. RAG vs. fine-tuning
2. **Data Preparation**: JSONL format for OpenAI and HuggingFace fine-tuning
3. **LoRA/PEFT**: Parameter-efficient fine-tuning with HuggingFace Transformers
4. **Evaluation**: BLEU, ROUGE metrics and LLM-as-judge evaluation patterns
5. **Deployment**: Building interactive web apps with Gradio
6. **Ethics**: Responsible AI use, bias, privacy, and transparency

### Workshop Recap

Over three days, we have covered the complete lifecycle of building LLM-powered applications:

- **Day 1**: Foundations -- Transformers, prompt engineering, OpenAI API
- **Day 2**: Knowledge -- Embeddings, vector databases, RAG, LangChain
- **Day 3**: Production -- Fine-tuning, evaluation, deployment, ethics

### Recommended Next Steps

1. Build a RAG application for your own research papers or documents
2. Experiment with prompt engineering patterns on your specific tasks
3. Try fine-tuning a small model (e.g., Phi-2) on domain-specific data
4. Deploy a prototype with Gradio and gather user feedback
5. Explore agent frameworks (LangGraph, CrewAI) for complex workflows
6. Stay current: follow arXiv, Hugging Face, and AI conference proceedings